In [ ]:
# ===========
# Exercise 3
# ===========

from pathlib import Path
import numpy as np
from scipy.signal import butter, filtfilt, welch
from DataProcessor import DataProcessor

ACTIVITY_NAMES = {0: "Unknown", 1: "Ruhen", 2: "Normales Gehen", 3: "Schnelles Gehen", 4: "Rennen"}

def total_acc_magnitude(x, y, z):
    mag = np.sqrt(x**2 + y**2 + z**2)
    mag = mag - np.mean(mag)
    return mag


def bandpass_filter(signal, fs, lowcut=0.5, highcut=5.0, order=4):
    nyquist = fs / 2.0
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band")
    filtered = filtfilt(b, a, signal)
    return filtered


def estimate_cadence(block, fs):
    corr = np.correlate(block, block, mode="full")
    corr = corr[len(corr) // 2 :]
    
    min_lag = int(fs / 3.5)
    max_lag = int(fs / 0.8)
    
    if max_lag >= len(corr):
        max_lag = len(corr) - 1
    if min_lag >= max_lag:
        return 0.0
    
    part = corr[min_lag:max_lag]
    if len(part) == 0:
        return 0.0
    
    peak_lag = np.argmax(part) + min_lag
    cadence = 60.0 / (peak_lag / fs)
    return cadence


def dominant_frequency(signal, fs):
    sig_filt = bandpass_filter(signal, fs)
    nperseg = min(2 * fs, len(sig_filt))
    freqs, pxx = welch(sig_filt, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
    mask = (freqs >= 0.5) & (freqs <= 3.5)
    
    if not np.any(mask):
        return 0.0
    
    idx = np.argmax(pxx[mask])
    return freqs[mask][idx]


def classify_activity(cadence):
    if cadence <= 0: # 0 = unknown
        return 0
    elif cadence < 60:
        return 1  # Ruhen
    elif cadence < 110:
        return 2  # Normales Gehen
    elif cadence < 140:
        return 3  # Schnelles Gehen
    else:
        return 4  # Rennen


def cadence_activity_algorithm(accelerationData, fs):
    """
    Input: accelerationData als Nx3 oder 3xN, fs = sampling frequency in Hz
    Output: cadence_vector, activity_vector (pro 5 Sekunden)
    """
    arr = np.array(accelerationData)
    
    # sowohl Nx3 als auch 3xN erlaubt
    if arr.ndim != 2:
        return [], []
    if arr.shape[1] == 3:
        acc = arr
    elif arr.shape[0] == 3:
        acc = arr.T
    else:
        return [], []

    x = acc[:, 0]
    y = acc[:, 1]
    z = acc[:, 2]

    block_samples = int(5 * fs)
    num_blocks = len(x) // block_samples

    cadence_vector = []
    activity_vector = []

    for i in range(num_blocks):
        start = i * block_samples
        end = start + block_samples

        mag = total_acc_magnitude(x[start:end], y[start:end], z[start:end])
        mag = bandpass_filter(mag, fs)
        cad = estimate_cadence(mag, fs)
        act = classify_activity(cad)
        cadence_vector.append(float(cad))
        activity_vector.append(int(act))

    return cadence_vector, activity_vector


def analyze_file(file_path):
    dp = DataProcessor("rawdata/X22/")
    dp.loadRawData(str(file_path))

    devices = dp.getDevices()
    if len(devices) == 0:
        return 0, [], []

    dp.loadRawDataDevice(devices[0])
    x = dp.dfAcc["x"].values
    y = dp.dfAcc["y"].values
    z = dp.dfAcc["z"].values
    fs = dp.fs

    accelerationData = np.column_stack((x, y, z))
    cadence_vector, activity_vector = cadence_activity_algorithm(accelerationData, fs)
    return fs, cadence_vector, activity_vector


# ------------------------------------------------------------
#Daten laden und analysieren
# ------------------------------------------------------------
paths = [
    Path("rawdata/X22/Excersice_2/rennen/"),
    Path("rawdata/X22/Excersice_2/normal_gehen/"),
    Path("rawdata/X22/Excersice_2/schnell_gehen/")
]

for dir_path in paths:
    print(f"\n=== {dir_path} ===")

    if not dir_path.exists():
        print("Verzeichnis nicht gefunden.")
        continue

    files = sorted(dir_path.rglob("*.pickle"))
    if len(files) == 0:
        print("Keine Dateien gefunden.")
        continue

    for file_path in files:
        print(f"\n  --- {file_path.name} ---")
        fs, cadences, activities = analyze_file(file_path)

        print(f"  Samplingrate: {fs} Hz")
        print(f"  Anzahl 5s-Bloecke: {len(cadences)}")
        if len(cadences) > 0:
            print(f"  Cadence (steps/min): {[f'{c:.1f}' for c in cadences]}")
            print(f"  Aktivitaeten: {[(a, ACTIVITY_NAMES[a]) for a in activities]}")
            print(f"  Mittlere Cadence: {np.mean(cadences):.1f} steps/min")
        else:
            print("  Keine vollstaendigen Bloecke.")


c:\Users\elyes\anaconda3\envs\DSP_Projekt\Lib\site-packages\vpython\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>


=== rawdata\X22\Excersice_2\rennen ===

  --- Hakim_rennnen_1.pickle ---
  Samplingrate: 200 Hz
  Anzahl 5s-Bloecke: 10
  Cadence (steps/min): ['99.2', '179.1', '65.2', '83.3', '131.9', '210.5', '210.5', '48.2', '57.7', '210.5']
  Aktivitaeten: [(2, 'Normales Gehen'), (4, 'Rennen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (3, 'Schnelles Gehen'), (4, 'Rennen'), (4, 'Rennen'), (1, 'Ruhen'), (1, 'Ruhen'), (4, 'Rennen')]
  Mittlere Cadence: 129.6 steps/min

  --- Hakim_rennnen_10.pickle ---
  Samplingrate: 200 Hz
  Anzahl 5s-Bloecke: 13
  Cadence (steps/min): ['73.6', '74.1', '73.6', '72.3', '69.4', '68.2', '69.0', '69.0', '68.2', '70.6', '72.3', '210.5', '210.5']
  Aktivitaeten: [(2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (2, 'Normales Gehen'), (4, 'Rennen'), (4, 'Rennen')]
  Mittlere Cadence: 92.4

In [ ]:

# ============================================================
# Exercise 4: Test des Algorithmus auf gelabelten Ordnerdaten
# ============================================================

from pathlib import Path
import numpy as np


# Diese Funktion gibt das richtige Label (Ground Truth) fuer eine Datei zurueck,
# basierend auf dem Ordnernamen.
def ground_truth_label(file_path):
    # Pfad in Kleinbuchstaben umwandeln, damit Gross-/Kleinschreibung egal ist
    text = str(file_path).lower().replace("\\", "/")
    name = file_path.name.lower()

    if "rennen" in text:
        return 4  # Rennen
    elif "schnell_gehen" in text or  "schnell_laufen" in text:
        return 3  # Schnelles Gehen
    elif "normal_gehen" in text or "gehen" in text or "normal_laufen":
        return 2  # Normales Gehen
    else:
        return 0  # Unbekannt

# ----------------------------------------------------------
# Schritt 1: Alle Testdateien sammeln
# ----------------------------------------------------------
all_files = []

for dir_path in paths:
    if dir_path.exists():
        # Alle .pickle Dateien im Ordner und Unterordnern suchen
        found = sorted(dir_path.rglob("*.pickle"))
        for f in found:
            all_files.append(f)

print(f"Gefundene Testdateien: {len(all_files)}")

# ----------------------------------------------------------
# Schritt 2: Jede Datei analysieren und Ergebnis speichern
# ----------------------------------------------------------

# Listen fuer die wahren Labels und die vorhergesagten Labels
y_true = []  # Ground Truth (aus Ordnername)
y_pred = []  # Vorhersage des Algorithmus

# Liste fuer eine Zusammenfassung pro Datei
per_file = []

for file_path in all_files:
    # Wahres Label aus dem Ordnernamen bestimmen
    true_label = ground_truth_label(file_path)

    # Algorithmus ausfuehren
    fs, cadence_vector, pred_blocks = analyze_file(file_path)

    # Datei ueberspringen wenn keine Bloecke gefunden wurden
    if len(pred_blocks) == 0:
        continue

    # Jeden Block einzeln in die Listen eintragen
    for p in pred_blocks:
        y_true.append(true_label)
        y_pred.append(p)

    # Genauigkeit fuer diese Datei berechnen
    correct = 0
    for p in pred_blocks:
        if p == true_label:
            correct = correct + 1
    file_acc = correct / len(pred_blocks)

    per_file.append((file_path, true_label, len(pred_blocks), file_acc))


# ----------------------------------------------------------
# Schritt 3: Auswertung ausgeben
# ----------------------------------------------------------

print(f"Ausgewertete Bloecke gesamt: {len(y_true)}")
print(f"Dateien mit mindestens 1 Block: {len(per_file)}")

if len(y_true) == 0:
    print("Keine auswertbaren Bloecke gefunden.")
else:
    # Gesamtgenauigkeit berechnen
    correct_total = 0
    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct_total = correct_total + 1
    accuracy = correct_total / len(y_true)

    # Confusion Matrix aufbauen: Zeilen = wahres Label, Spalten = vorhergesagtes Label
    labels = [0, 1, 2, 3, 4]
    cm = np.zeros((len(labels), len(labels)), dtype=int)

    for i in range(len(y_true)):
        zeile = labels.index(y_true[i])
        spalte = labels.index(y_pred[i])
        cm[zeile][spalte] = cm[zeile][spalte] + 1

    print(f"\nAccuracy (blockbasiert): {accuracy:.3f}")
    print("Labels:", labels, "->", [ACTIVITY_NAMES[l] for l in labels])
    print("\nConfusion Matrix (Zeilen=true, Spalten=pred):")
    print(cm)

    # Recall pro Klasse berechnen (Wie viel % der echten Klasse X wurde richtig erkannt?)
    print("\nEinfache Klassen-Recall-Werte:")
    for i in range(len(labels)):
        label = labels[i]
        total_true = 0
        for j in range(len(labels)):
            total_true = total_true + cm[i][j]

        if total_true == 0:
            recall = 0.0
        else:
            recall = cm[i][i] / total_true

        print(f"{label} ({ACTIVITY_NAMES[label]}): Recall={recall:.3f}")

    # Erste 15 Dateien einzeln ausgeben
    print("\nDateiuebersicht (erste 15):")
    for idx in range(min(15, len(per_file))):
        file_path, gt_label, n_blocks, file_acc = per_file[idx]
        print(
            f"{file_path} | GT={gt_label} ({ACTIVITY_NAMES[gt_label]}) | "
            f"Bloecke={n_blocks} | Datei-Accuracy={file_acc:.3f}"
        )

Gefundene Testdateien: %f 79
Ausgewertete Bloecke gesamt: 565
Dateien mit mindestens 1 Block: 79

Accuracy (blockbasiert): 0.283
Labels: [0, 1, 2, 3, 4] -> ['Unknown', 'Ruhen', 'Normales Gehen', 'Schnelles Gehen', 'Rennen']

Confusion Matrix (Zeilen=true, Spalten=pred):
[[  0   0   0   0   0]
 [  0   0   0   0   0]
 [  0  30 115  18  40]
 [  0  35 138  20  51]
 [  0  17  66  10  25]]

Einfache Klassen-Recall-Werte:
0 (Unknown): Recall=0.000
1 (Ruhen): Recall=0.000
2 (Normales Gehen): Recall=0.567
3 (Schnelles Gehen): Recall=0.082
4 (Rennen): Recall=0.212

Dateiuebersicht (erste 15):
rawdata\X22\Excersice_2\rennen\Hakim_rennnen_1.pickle | GT=4 (Rennen) | Bloecke=10 | Datei-Accuracy=0.400
rawdata\X22\Excersice_2\rennen\Hakim_rennnen_10.pickle | GT=4 (Rennen) | Bloecke=13 | Datei-Accuracy=0.154
rawdata\X22\Excersice_2\rennen\Hakim_rennnen_2.pickle | GT=4 (Rennen) | Bloecke=1 | Datei-Accuracy=0.000
rawdata\X22\Excersice_2\rennen\Hakim_rennnen_3.pickle | GT=4 (Rennen) | Bloecke=7 | Datei-Ac